# Machine Learning Mathematics

It's important to identify some of the computational mathematics that are used in some of our models, aside from the actual models themselves. 

## Logarithmic Likelihoods & Fisher Information

### Introduction

A likelihood function $L(\boldsymbol{\theta})$ is the joint probability of all data points $X$ in a set $\{x_1,x_2,..,x_n\}$, where $\boldsymbol{\theta}$ is vector encoding the model parameters:
$$L(\boldsymbol{\theta}) = \prod_{i=1}^{n} f(x_i \mid \boldsymbol{\theta})$$
In order to solve for any model parameters, the goal is to find the $\boldsymbol{\theta}$ that maximizes $L(\boldsymbol{\theta})$. In other words, we want to solve for the parameters that actually yield our data points. However, working with probabilities entails two issues: For one, multiplying probabilities $f(x_i \mid \boldsymbol{\theta}) \in [0, 1]$ results in incredibly low numbers, and differentiating these expressions in order to find the maximum $\boldsymbol{\theta}$ requires excessive use of the product rule. So, we instead take the log-likelihood:
$$\ell(\boldsymbol{\theta}) = \ln L(\boldsymbol{\theta}) = \ln \left( \prod_{i=1}^{n} f(x_i \mid \boldsymbol{\theta}) \right) = \sum_{i=1}^{n} \ln f(x_i \mid \boldsymbol{\theta})$$
This is valid, since both functions are monotonically increases. Thus: 
$$\arg\max_{\boldsymbol{\theta}} L(\boldsymbol{\theta}) = \arg\max_{\boldsymbol{\theta}} \ell(\boldsymbol{\theta})$$
What this means, is that in both equations, $L(\boldsymbol{\theta})$ corresponds to the maximum. And so, we can effectively compress our original product sum into a clean sum that also differentiates easily.

### Application to Multivariable Calculus

When parameters are multidimensional, $\ell(\boldsymbol{\theta})$ yields a high-dimensional surface. The gradient function of $\ell(\boldsymbol{\theta})$ with respect to every parameter vector. This gradient vector is called the score function $\mathbf{s}(\boldsymbol{\theta})$:
$$\mathbf{s}(\boldsymbol{\theta}) = \nabla_{\boldsymbol{\theta}} \ell(\boldsymbol{\theta}) = \begin{bmatrix} \frac{\partial \ell}{\partial \theta_1} \\ \frac{\partial \ell}{\partial \theta_2} \\ \vdots \\ \frac{\partial \ell}{\partial \theta_d} \end{bmatrix} = \sum_{i=1}^{n} \nabla_{\boldsymbol{\theta}} \ln f(x_i \mid \boldsymbol{\theta})$$
Remember, the number of partial derivatives we take is equivalent to the number of parameters we have. If $\theta=\begin{bmatrix}
\alpha\\
\beta
\end{bmatrix}$, we only take two partial derivatives. And of course, to find possible candidates for parameter estimations, we solve for where the gradient is equal to zero. But sometimes, we may use gradient descent algorithms to optimize these estimations:
$$\boldsymbol{\theta}^{(t+1)} = \boldsymbol{\theta}^{(t)} + \alpha \nabla_{\boldsymbol{\theta}} \ell\left(\boldsymbol{\theta}^{(t)}\right)$$
To verify where every point is either a maximum or a minimum, we must derive a Hessian Matrix $\boldsymbol{H}$:
$$\mathbf{H}(\boldsymbol{\theta}) = \nabla^2_{\boldsymbol{\theta}} \ell(\boldsymbol{\theta}) = \begin{bmatrix} \frac{\partial^2 \ell}{\partial \theta_1^2} & \frac{\partial^2 \ell}{\partial \theta_1 \partial \theta_2} & \cdots & \frac{\partial^2 \ell}{\partial \theta_1 \partial \theta_d} \\ \frac{\partial^2 \ell}{\partial \theta_2 \partial \theta_1} & \frac{\partial^2 \ell}{\partial \theta_2^2} & \cdots & \frac{\partial^2 \ell}{\partial \theta_2 \partial \theta_d} \\ \vdots & \vdots & \ddots & \vdots \\ \frac{\partial^2 \ell}{\partial \theta_d \partial \theta_1} & \frac{\partial^2 \ell}{\partial \theta_d \partial \theta_2} & \cdots & \frac{\partial^2 \ell}{\partial \theta_d^2} \end{bmatrix}$$
Note, again, if $\theta=\begin{bmatrix}
\alpha\\
\beta
\end{bmatrix}$, we only have a $4\times4$ matrix. Note that second-order optimization algorithms use the inverse of the Hessian for step sizes:
$$\boldsymbol{\theta}^{(t+1)} = \boldsymbol{\theta}^{(t)} - \mathbf{H}^{-1}\left(\boldsymbol{\theta}^{(t)}\right) \nabla_{\boldsymbol{\theta}} \ell\left(\boldsymbol{\theta}^{(t)}\right)$$
The Fisher Information is important, as it links the curvature of a log-likelihood function to parameter uncertainty. The Fisher Information is defined as:
$$\mathbf{I}(\boldsymbol{\theta}) = -\mathbb{E} \left[ \nabla^2_{\boldsymbol{\theta}} \ell(\boldsymbol{\theta}) \right]$$
Large negative values in the Hessian matrix means the likelihood falls away from where the optimal parameters are, highlighting a low parameter uncertainty. On the other hand, a more flat Hessian implies there is not much of a likelihood falloff, suggesting high parameter variance. 

## BFGS Algorithm

This is the most efficient means of parameter optimization. The first step involves calculating the gradient vector $\mathbf{g}_k = \nabla_{\boldsymbol{\theta}} \ell(\boldsymbol{\theta}_k)$, and then multiplying it with the inverse Hessian matrix $\mathbf{B}_k$:
$$\mathbf{p}_k = \mathbf{B}_k \mathbf{g}_k$$
This is matrix is comprised of the possible candidates for our step length $\alpha_k$. The algorithm performs a quick 1D search to find an algorithm that satisfies the Wolfe Conditions:

Then, we actually can update the parameters:
$$\boldsymbol{\theta}_{k+1} = \boldsymbol{\theta}_k + \alpha_k \mathbf{p}_k$$
Now we have to update the inverse Hessian approximation. Calculate two vectors, the parameter shift vector $\mathbf{s}_k = \boldsymbol{\theta}_{k+1} - \boldsymbol{\theta}_k$, and the gradient shift vector $\mathbf{y}_k = \mathbf{g}_{k+1} - \mathbf{g}_k$. The updated inverse Hessian is then:
$$\mathbf{B}_{k+1} = (\mathbf{I} - \rho_k \mathbf{s}_k \mathbf{y}_k^T) \mathbf{B}_k (\mathbf{I} - \rho_k \mathbf{y}_k \mathbf{s}_k^T) + \rho_k \mathbf{s}_k \mathbf{s}_k^T \quad \text{where } \rho_k = \frac{1}{\mathbf{y}_k^T \mathbf{s}_k}$$
This is inherently faster than the naive inverse Hessian update, and more accurate than using a fixed learning rate. 

## Levenberg-Marquardt Algorithm

This is an optimization technique used to specifically solve non-linear least-squares problems. If data points do not depend linearly on parameters, you must iteratively search for the most optimal parameters that minimize the sum of squared errors between predicted and observed data:
$$S(\boldsymbol{\beta}) = \sum_{i=1}^{m} \left[ y_i - f(x_i, \boldsymbol{\beta}) \right]^2 = \Vert{}\boldsymbol{r}(\boldsymbol{\beta})\Vert{}^2$$
here, $\boldsymbol{r}\boldsymbol{(\beta)}$ is a vector of residuals where $r_i(\boldsymbol{\beta}) = y_i - f(x_i, \boldsymbol{\beta})$, and $\boldsymbol{\beta}$ is the parameter vector we are trying to estimate. The main solving mechanism revolves around solving several augmented linear equations. First, we must choose an initial parameter guess $\boldsymbol{\beta_0}$, and an initial dampening factor $\lambda$ (typically $10^{-3}$ or $10^{-2}$). Then, set convergence thresholds for residual error $\epsilon_1$, parameter change $\epsilon_2$, and gradient norm $\epsilon_3$. 

Now, for a current estimate $\beta_k$, compute the residual vector $\boldsymbol{r}(\boldsymbol{\beta}_k)$. Then, compute the Jacobian matrix $\boldsymbol{J}$, where $J_{ij} = \frac{\partial r_i}{\partial \beta_j}$. Now, compute the gradient vector $\boldsymbol{g} = \boldsymbol{J}^T \boldsymbol{r}$ and approximate the Hessian matrix as $\boldsymbol{H} \approx \boldsymbol{J}^T \boldsymbol{J}$. 

Now, solve for the parameter update $\boldsymbol{\delta_k}$:
$$(\boldsymbol{J}^T \boldsymbol{J} + \lambda \boldsymbol{D}) \boldsymbol{\delta}_k = \boldsymbol{J}^T \boldsymbol{r}$$
Note that $\boldsymbol{D}$ is either the identity matrix or the diagonal of $\boldsymbol{J^T}\boldsymbol{J}$ depending on the scaling differences between the parameters. 

Now, evaluate the new parameter $\beta_{trial}=\beta_{k}+\boldsymbol{\delta_k}$. If $S(\beta_{trial})<S(\beta_{k})$, this implies success. We can accept the new parameter and then set $\lambda=\lambda/10$. Otherwise, we reject the step and increase $\lambda=\lambda \times 10$

Now, we check our final thresholds:
- $\Vert{}\boldsymbol{g}\Vert{} < \epsilon_3$ (the gradient is sufficiently close to zero)
- $\Vert{}\boldsymbol{\delta}_k\Vert{} / \Vert{}\boldsymbol{\beta}_k\Vert{} < \epsilon_2$ (parameter change isn't extreme)
- $\vert{}S(\boldsymbol{\beta}_{k+1}) - S(\boldsymbol{\beta}_k)\vert{} < \epsilon_1$ (improvement in cost function is approaching zero)

If all of these are true, then we can terminate the optimization. 

## Kalman Filtering

This is used to estimate the true, unobserved state of a dynamic system over time, through measurements with a relatively high degree of noise. The Kalman filter specifically computes a weighted average of these measurements and model predictions. 

A the model predictions have an intrinsic noise $Q$, and all measurements inherently have a noise $R$. These two sources of information are balanced using the Kalman gain $K_t$:
$$\text{New Estimate} = \text{Model Prediction} + K_t \cdot (\text{Sensor Measurement} - \text{Model Prediction})$$
If the model is more accurate, $K_t$ decreases. If the sensor is more accurate, then $K_t$ increases. \

A standard Kalman filter assumes state $x_t$ and measurement $z_t$ as following:
$$\begin{aligned} \text{State Equation:} \quad & x_t = F_t x_{t-1} + B_t u_t + w_t, \quad w_t \sim \mathcal{N}(0, Q_t) \\ \text{Measurement Equation:} \quad & z_t = H_t x_t + v_t, \quad v_t \sim \mathcal{N}(0, R_t) \end{aligned}$$
Where:
- $x_t$ is the unobserved true state vector
- $z_t$ is the observed measurement vector
- $F_t$ is the state's transition matrix
- $H_t$ is effectively the emission matrix
- $Q_t$ and $R_t$ are the covariance matrices for state noise and measurement noise, respectively



## Lagrangian Multipliers